# Exercise 5
In this exercise you will:
- Train regression and classification models using random forests.
- Visualize individual trees
- Check feature importances
- Get general classification metrics including area under the curve.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, roc_curve, auc # different metrics 
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor # RF model
from sklearn.tree import plot_tree # function to visualize a decision tree

<Br><Br>
### Load the data

In [ ]:
data = pd.read_csv('wines.csv') # Note: there are missing values and many more white wines than red wines...
data

<Br><Br>
### Clean the dataset

In [ ]:
# Are there missing values? Here, .sum() sums all samples with missing values PER COLUMN
data.isna().sum()

In [ ]:
data = data.dropna() # remove samples with missing values (we could also replace missing values by column averages)
data.shape # check how many samples (rows) are there after removing missing values (originally there were 6497 samples)

<Br><Br>
### Get white and red wines separately

In [ ]:
data['type'] == 'white' # This is a mask to get where are the white wines

In [ ]:
mask_white = data['type'] == 'white' # save the mask as object mask_white
mask_white # check the mask

In [ ]:
mask_red = data['type'] == 'red' # save the mask as object mask_red
mask_red # check the mask

In [ ]:
mask_white.sum() , mask_red.sum() # total number of white and red wines (= number of Trues in each mask) 
# The dataset is UNBALANCED!! 4870 whites + 1593 reds

In [ ]:
# let's separate whites and reds into two different dataframes
whites = data[mask_white]
reds = data[mask_red]

In [ ]:
# get (randomly) only 1593 white wines (to achieve a balanced dataset)
whites = whites.sample(n = 1593)

<Br><Br>
### Give a look at the statistics

In [ ]:
whites.describe() # note that statistics were not shown for the first column (='white')

In [ ]:
reds.describe()

In [ ]:
# check histograms of individual columns 
whites.hist(figsize=(15,15));

In [ ]:
# check histograms of individual columns 
reds.hist(figsize=(15,15));
# the histograms of whites and reds seem to be very similar (=features are balanced)

<Br><Br>
### Reunify clean + balanced subsets

In [ ]:
# final DataFrame with balanced reds and whites, no missing values
data_cured = pd.concat([reds, whites]) # reds and whites are concatenated (row-wise) into DataFrame "data_cured" 

In [ ]:
data_cured

In [ ]:
# Let's shuffle the DataFrame
data_cured = data_cured.sample(frac = 1) # frac = fraction of rows to be shuffled (or "sampled"). Here, 1 = 100% of samples (rows)

In [ ]:
data_cured

<Br><Br>
# Prediction of wine type (class = "white" or "red")
<Br><Br>

In [ ]:
# the target vector will be the column "type"
X = data_cured.drop(['type'], axis = 1) # features are all columns except the target column
y = data_cured['type'] # classes (=white or red)

In [ ]:
# check the features
X

In [ ]:
# check the target property. First column = index of the sample in the original(imported) DataFrame
y

<Br><Br>
### Split the dataset


In [ ]:
# we only use random_state to be able to split again X and y to generate precisely the same samples
# in X_train, y_train, etc.  Note that the function train_test_split() shuffles all rows of the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y) # stratify: get balanced subsets!

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape 

In [ ]:
(y_test == 'white').sum() # you have precisely half of the test set for each class! (your test set is balanced)

<Br><Br>
### Preprocessing

In [ ]:
# Let's preprocess the features
scaler = StandardScaler()
scaler.fit(X_train) #mean and std of each feature of X_train are stored inside object "scaler"
X_train_s = scaler.transform(X_train) # convert X_train into zero mean and unit variance
X_test_s = scaler.transform(X_test) # convert X_test into zero mean and unit variance

<Br><Br>
### Train the RF model, get accuracy

In [ ]:
rf = RandomForestClassifier() # create the model (with 100 trees by default)
rf.fit(X_train_s, y_train) # train the model (optimize splitting parameters for all (100) trees)

pred = rf.predict(X_test_s) # perform predictions

acc = rf.score(X_test_s, y_test) # number of correct classifications / total number of classifications

print(f'accuracy: {acc*100:.2f}%')

<Br><Br>
### Show more metrics

In [ ]:
report = classification_report(y_test, pred) 
print(report) # without the 'stratify' parameter used during dataset splitting, the 'support' below would be different for both classes

In [ ]:
confusion_matrix(y_test, pred) # from above, "white" is class 0 (first row below), 'red' is the second row

<Br><Br>
### Feature importances

In [ ]:
# check which features are more important
feature_names = X.columns # ".columns" attribute of DataFrame object X has the names of all columns
importances = rf.feature_importances_ # get feature importances from the trained model

In [ ]:
plt.bar(feature_names, height=importances)
plt.xticks(rotation=90);
# note: quality is not important for classification: white and red wines can be of high quality! 

<Br><Br>
### Area under the curve (classification metrics)

In [ ]:
# let's see now the Receiver Operating Curve (ROC) and the corresponding area (= AUC)

decision_function = rf.predict_proba(X_test_s)[:,1] # change 1 by 0 for using the negative class
# This info is used to build the ROC curve, where different thresholds are using while accuracy is monitored

TPR, TNR, thresholds = roc_curve(y_test, decision_function, pos_label='red') 
# TNR (= True Negative Rate) = TN / (TN + FP)
# TPR (= True Positive Rate) = TP / (TP + FN)

area_under_curve = auc(TNR, TPR) # area under the ROC curve (function "auc()", imported from sklearn)

# plot the ROC curve
plt.plot(TNR, TPR,'bo-')
plt.title(f'Area under the ROC curve: {area_under_curve:.4f} ')
plt.xlabel('True Negative Rate')
plt.ylabel('True Positive Rate');

<Br><Br>
### Let's check some trees

In [ ]:
# Let's decrease the maximum depth of the tree to facilitate visualization:
rf = RandomForestClassifier(max_depth=3)
rf.fit(X_train_s, y_train)

In [ ]:
tree = rf.estimators_[0] # Let's visualize the first tree

plt.figure(figsize=(20, 10))
plot_tree(
    tree,
    feature_names=feature_names,  # List of your feature names
    class_names=['white', 'red'],      # List of your class labels
    filled=True,
    rounded=True
)
plt.title("Decision Tree from the Random Forest", fontsize=20);
# plt.savefig('tree1.pdf', dpi=600) # uncomment this line to save your tree

In [ ]:
# Visualize the last tree


    # Your code here



<Br><Br>
# TASK: Instead of predicting the wine type, predict the wine quality using all other features (show the final accuracy for this multi-class classification task).
<Br><Br>

In [ ]:
# To use column "type" as a feature in our classification, convert "red" = 0 and "white" = 1, otherwise
# we cannot include this column in our model (we need numbers!)

myencoder = LabelEncoder()

col_type = data_cured['type'] # column I want to encode as 0 and 1
col_encoded = myencoder.fit_transform(col_type)

data_cured['type'] = col_encoded # update/redefine column "type" with new (encoded) values

In [ ]:
myencoder.classes_ # check classes ('red' = 0, 'white' = 1) 

In [ ]:
# check the data again (your target property is the column 'quality'
data_cured.head() # now column "type" contains only 0's and 1's and is a simple feature

In [ ]:
# Check how balanced is the dataset: How many samples each class has? You can make a histogram of column 'quality'. 
# If any class has less than 20 samples, eliminate these samples.
# Also note: The model accuracy will be worse for classes with fewer elements...


    # Your code here



In [ ]:
# Define your dataset: features (X) and target (y)


    # Your code here




In [ ]:
# Split your dataset (20% for the test set)


    # Your code here




In [ ]:
# Preprocess the features


    # Your code here



In [ ]:
# Create and train the RF model (use default hyperparameters)


    # Your code here



In [ ]:
# Perform predictions on the test set


    # Your code here



In [ ]:
# Calculate the accuracy (for predictions on the test set)


    # Your code here



In [ ]:
# Show a classification report


    # Your code here



In [ ]:
# Get a confusion matrix and visualize it using the seaborn library 


    # Your code here (generate the confusion matrix to be used in the last line


import seaborn as sns # library based on matplotlib to easily make nice plots
classes = ['4','5','6','7','8'] # CHeck that your classes are here
sns.heatmap(confusion_matrix, yticklabels=classes, xticklabels=classes); # Note: In the plot below, classes are RENUMBERED to 0, 1, 2, ...

<Br><Br>
# TASK: Repeat the previous task using regression instead of classification and show regression metrics and a parity plot.
<Br><Br>

<Br><Br>
# HOMEWORK
Make a binary classification of the wine type (white or red). Use a for loop to train a RF model using different number of trees (e.g. from 1 to 30) and plot the final classification accuracy as a function of the number of trees. Important: use the same random state inside the RF model to keep consistency.
<Br><Br>
<Br><Br>

In [ ]:


    # Your code here



<Br><Br>
# Challenge
Repeat the procedure above using 10 different random states inside the RF model for each number of trees used, so that you can plot the results showing error bars corresponding to the standard deviation of the accuracy.
<Br><Br>
<Br><Br>

In [ ]:


    # Your code here

